<a href="https://colab.research.google.com/github/reskitaamelia/Skripsi-Reskita-Amelia/blob/main/imageWatermarking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install numba reedsolo

import cv2
import numpy as np
import os
import random
import math
import json
import numba
import zlib
import secrets

from scipy.ndimage import gaussian_filter
from matplotlib import pyplot as plt
from skimage.metrics import structural_similarity
from reedsolo import RSCodec, ReedSolomonError

from google.colab.patches import cv2_imshow
from google.colab import drive
drive.mount('/content/drive/')

# Embedding Section

In [ ]:
# functions for host image

def convert_host_image(input_folder, output_folder, size=(1280, 1280), color_mode='color'):
  if not os.path.exists(output_folder):
    os.makedirs(output_folder)

  if not os.path.exists(input_folder):
    raise ValueError(f'Input folder {input_folder} not found')

  try:
    #  iterate over all files in the input folder
    for filename in os.listdir(input_folder):
      try:
        input_image_path = os.path.join(input_folder, filename)

        #  check if the file is an image
        if os.path.isfile(input_image_path) and filename.lower().endswith(('.png', '.jpg', '.jpeg')):
          #  read the input image
          image = cv2.imread(input_image_path)

          if image is None:
            print(f"Failed to read image: {input_image_path}")
            continue

          if color_mode != 'color':
            processed_img = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
          else:
            processed_img = image

          # define the output image path
          resized_image = cv2.resize(processed_img, size)
          output_image_path = os.path.join(output_folder, filename)

          # save the processed image
          cv2.imwrite(output_image_path, resized_image)
          print(f"Processed image saved at: {output_image_path}")
      except Exception as file_error:
        print(f'Error processing file {filename}: {file_error}')
        continue
  except Exception as folder_error:
    print(f'Error processing folder {input_folder}: {folder_error}')

In [ ]:
# function for watermark

def convert_wm(input_folder, output_folder, size=(64, 64), threshold=128):
  # validate input/output paths
  if not os.path.exists(input_folder):
    raise FileNotFoundError(f"Input folder '{input_folder}' does not exist.")
  os.makedirs(output_folder, exist_ok=True)

  # iterate over all files in the input folder
  for filename in os.listdir(input_folder):
    input_image_path = os.path.join(input_folder, filename)

    # check if the file is an image
    if os.path.isfile(input_image_path) and filename.lower().endswith(('.png', '.jpg', '.jpeg')):
      # read the input image
      wm = cv2.imread(input_image_path, cv2.IMREAD_GRAYSCALE)

      if wm is None:
        print(f"Failed to read image: {input_image_path}")
        continue

      # resize the image
      resized_wm = cv2.resize(wm, size, interpolation=cv2.INTER_NEAREST)

      # binarize the watermark
      _, binary_wm = cv2.threshold(resized_wm, threshold, 1, cv2.THRESH_BINARY)

      # define the output image path
      output_wm_path = os.path.join(output_folder, filename)

      # save the processed image
      cv2.imwrite(output_wm_path, binary_wm * 255)
      print(f"Processed watermark saved at: {output_wm_path}")

In [ ]:
host_input_dir = '/content/drive/My Drive/Tugas Akhir/Dataset'
host_output_dir = '/content/drive/My Drive/Tugas Akhir/Dataset Resized'
wm_input_dir = '/content/drive/My Drive/Tugas Akhir/Watermark'
wm_output_dir = '/content/drive/My Drive/Tugas Akhir/Watermark Resized'

print(f"Starting image processing...")
print(f"Processing host images...")
convert_host_image(host_input_dir, host_output_dir)

print('')
print(f"Processing watermarks...")
convert_wm(wm_input_dir, wm_output_dir)

In [ ]:
import os

print(os.listdir('/content/drive/My Drive/Tugas Akhir/Dataset Resized'))
print(os.listdir('/content/drive/My Drive/Tugas Akhir/Watermark Resized'))

In [ ]:
host_img_dir1 = '/content/drive/My Drive/Tugas Akhir/Dataset Resized/citra1.jpg'
host_img_dir2 = '/content/drive/My Drive/Tugas Akhir/Dataset Resized/citra2.jpg'
host_img_dir3 = '/content/drive/My Drive/Tugas Akhir/Dataset Resized/citra3.jpg'
host_img_dir4 = '/content/drive/My Drive/Tugas Akhir/Dataset Resized/citra4.jpg'
host_img_dir5 = '/content/drive/My Drive/Tugas Akhir/Dataset Resized/citra5.jpg'
host_img_dir6 = '/content/drive/My Drive/Tugas Akhir/Dataset Resized/citra6.jpg'

wm_dir1 = '/content/drive/My Drive/Tugas Akhir/Watermark Resized/qr1.png'
wm_dir2 = '/content/drive/My Drive/Tugas Akhir/Watermark Resized/qr2.png'
wm_dir3 = '/content/drive/My Drive/Tugas Akhir/Watermark Resized/qr3.png'
wm_dir4 = '/content/drive/My Drive/Tugas Akhir/Watermark Resized/qr4.png'
wm_dir5 = '/content/drive/My Drive/Tugas Akhir/Watermark Resized/qr5.png'
wm_dir6 = '/content/drive/My Drive/Tugas Akhir/Watermark Resized/qr6.png'

host_img = cv2.imread(host_img_dir2)
wm = cv2.imread(wm_dir2, cv2.IMREAD_GRAYSCALE)

## Transform Process

### DWT

In [ ]:
@numba.njit
def haar(signal):
  n = len(signal)
  if n % 2 != 0:
    raise ValueError("Signal length must be even")

  # low pass and high pass coeffs
  low = np.zeros(n // 2)
  high = np.zeros(n // 2)

  for i in range(n // 2):
    # haar wavelet scaling and wavelet func
    low[i] = (signal[2*i] + signal[2*i + 1]) / np.sqrt(2)
    high[i] = (signal[2*i] - signal[2*i + 1]) / np.sqrt(2)

  return low, high

@numba.njit
def inv_haar(low, high):
  n = len(low) + len(high)
  signal = np.zeros(n)

  for i in range(len(low)):
    # inverse haar wavelet transform
    signal[2*i] = (low[i] + high[i]) / np.sqrt(2)
    signal[2*i + 1] = (low[i] - high[i]) / np.sqrt(2)

  return signal

def dwt_2d(image):
  rows, cols = image.shape

  # apply dwt to each row
  temp = np.zeros_like(image)
  for i in range(rows):
    low, high = haar(image[i, :])
    temp[i, :cols//2] = low
    temp[i, cols//2:] = high

  # apply dwt to each col
  result = np.zeros_like(temp)
  for j in range(cols):
    low, high = haar(temp[:, j])
    result[:rows//2, j] = low
    result[rows//2:, j] = high

  LL = result[:rows//2, :cols//2]
  LH = result[:rows//2, cols//2:]
  HL = result[rows//2:, :cols//2]
  HH = result[rows//2:, cols//2:]

  return LL, LH, HL, HH

def idwt_2d(LL, LH, HL, HH):
  rows, cols = LL.shape

  print(f"Single-level IDWT shapes: LL:{LL.shape}, LH:{LH.shape}, HL:{HL.shape}, HH:{HH.shape}")

  if not (LL.shape == LH.shape == HL.shape == HH.shape):
    raise ValueError(f"All subbands must have the same shape for single-level IDWT. Got: "
                    f"LL:{LL.shape}, LH:{LH.shape}, HL:{HL.shape}, HH:{HH.shape}")

  # reconstruct freq domain
  temp = np.zeros((rows * 2, cols * 2))
  temp[:rows, :cols] = LL
  temp[:rows, cols:cols*2] = LH
  temp[rows:rows*2, :cols] = HL
  temp[rows:rows*2, cols:cols*2] = HH

  # apply idwt to each col
  result = np.zeros_like(temp)
  for j in range(cols * 2):
    result[:, j] = inv_haar(temp[:rows, j], temp[rows:, j])

  # apply idwt to each row
  final = np.zeros_like(result)
  for i in range(rows * 2):
    final[i, :] = inv_haar(result[i, :cols], result[i, cols:])

  return final

def apply_dwt(image, levels=1):
  if len(image.shape) != 2:
    raise ValueError("Input image must be 2D")

  rows, cols = image.shape

  # Check if dimensions are divisible by 2^levels
  div = 2 ** levels
  if rows % div != 0 or cols % div != 0:
    raise ValueError(f"Image dimensions must be divisible by {div} for {levels}-level DWT. "
                    f"Current dimensions: {rows}x{cols}")

  print(f"Applying {levels}-level DWT to {rows}x{cols} image")

  current = image.astype(np.float32)
  subbands = {}

  # apply dwt levels
  for level in range(levels):
    print(f"Level {level+1}: {current.shape} → ", end="")
    LL, LH, HL, HH = dwt_2d(current)
    print(f"subbands of {LL.shape}")

    # Store subbands for this level
    subbands[f'LH{level+1}'] = LH
    subbands[f'HL{level+1}'] = HL
    subbands[f'HH{level+1}'] = HH

    current = LL

  subbands[f"LL{levels}"] = current

  return subbands

def apply_idwt(subbands, ori_shape, levels=1):
  print(f"IDWT: Starting {levels}-level reconstruction")

  current = subbands[f'LL{levels}']
  print(f"Starting with LL subband of shape: {current.shape}")

  # apply idwt for each level
  for level in range(levels, 0, -1):
    LH = subbands[f'LH{level}']
    HL = subbands[f'HL{level}']
    HH = subbands[f'HH{level}']

    print(f"Level {level} reconstruction: LL:{current.shape}, LH:{LH.shape}, HL:{HL.shape}, HH:{HH.shape}")

    current = idwt_2d(current, LH, HL, HH)
    print(f"IDWT Level {level}: → {current.shape}")

  return current

In [ ]:
def split_HL(subbands, block_size=8):
  HL = subbands["HL1"].astype(np.float32)
  h, w = HL.shape

  assert h == 640 and w == 640, f"Unexpected HL1 dimensions: {HL.shape}. Expected: (640, 640)"
  # must be divisible by 8
  assert h % block_size == 0 and w % block_size == 0, \
    f"HL1 dimensions {HL.shape} not divisible by block size {block_size}"

  HL_blocks = []
  block_pos = []
  rows, cols = h // block_size, w // block_size

  for i in range(rows):
    for j in range(cols):
      block = HL[i*block_size:(i+1)*block_size,
                 j*block_size:(j+1)*block_size]
      HL_blocks.append(block.copy())
      block_pos.append((i, j))

  print(f"Created {len(HL_blocks)} blocks of size {block_size}x{block_size}")
  print(f"Block grid dimensions: {rows}x{cols}")

  return HL_blocks, HL.shape, block_pos

def recons_HL(dct_blocks, modif_blocks, selected_idx,
              block_pos, HL_shape, block_size=8):
  print(f"\nReconstructing complete HL subband...")
  print(f"  - Total blocks: {len(dct_blocks)}")
  print(f"  - Modified blocks: {len(modif_blocks)}")
  print(f"  - Unmodified blocks: {len(dct_blocks) - len(modif_blocks)}")
  print(f"  - Target HL shape: {HL_shape}")

  recons_blocks = []

  selected_idx_set = set(selected_idx)
  modif_block_map = {idx: block for idx, block in zip(selected_idx, modif_blocks)}

  # process each block
  for i, dct_block in enumerate(dct_blocks):
    if i in selected_idx_set:
      # use modif block
      current_dct_block = modif_block_map[i]
    else:
      # use ori block
      current_dct_block = dct_block

    spatial_block = idct_2d(current_dct_block)
    recons_blocks.append(spatial_block)

    # progress indicator
    if (i + 1) % 1000 == 0:
      print(f"  Processed {i + 1}/{len(dct_blocks)} blocks")

  recons_HL = np.zeros(HL_shape, dtype=np.float32)
  print(f"  - Placing blocks back into HL subband...")
  for block, (i, j) in zip(recons_blocks, block_pos):
    row_start = i * block_size
    row_end = (i + 1) * block_size
    col_start = j * block_size
    col_end = (j + 1) * block_size

    recons_HL[row_start:row_end, col_start:col_end] = block

  print(f"  - HL subband reconstruction completed!")
  print(f"  - Reconstructed HL shape: {recons_HL.shape}")

  return recons_HL

### DCT

In [ ]:
@numba.njit
def dct_2d(block):
  N = block.shape[0]
  assert block.shape == (N, N), f"Expected square block, got {block.shape}"

  # initialize dct coeff
  dct_block = np.zeros((N, N), dtype=np.float32)

  # compute dct coeffs
  for u in range(N):
    for v in range(N):
      # calculate normalization factors C(u) and C(v)
      Cu = 1.0/np.sqrt(N) if u == 0 else np.sqrt(2.0/N)
      Cv = 1.0/np.sqrt(N) if v == 0 else np.sqrt(2.0/N)

      sum_val = 0.0
      for m in range(N):
        for n in range(N):
          cos_u = np.cos((2*m + 1) * u * np.pi / (2*N))
          cos_v = np.cos((2*n + 1) * v * np.pi / (2*N))
          sum_val += block[m, n] * cos_u * cos_v

      dct_block[u, v] = Cu * Cv * sum_val

  return dct_block

@numba.njit
def idct_2d(dct_block):
  N = dct_block.shape[0]
  assert dct_block.shape == (N, N), f"Expected square block, got {dct_block.shape}"

  # initialize spatial domain block
  spatial_block = np.zeros((N, N), dtype=np.float32)

  # compute idct
  for m in range(N):
    for n in range(N):
      sum_val = 0.0
      for u in range(N):
        for v in range(N):
          # calculate normalization factors
          Cu = 1.0/np.sqrt(N) if u == 0 else np.sqrt(2.0/N)
          Cv = 1.0/np.sqrt(N) if v == 0 else np.sqrt(2.0/N)

          cos_u = np.cos((2*m + 1) * u * np.pi / (2*N))
          cos_v = np.cos((2*n + 1) * v * np.pi / (2*N))
          sum_val += dct_block[u, v] * Cu * Cv * cos_u * cos_v

      spatial_block[m, n] = sum_val

  return spatial_block

def apply_dct(HL_blocks):
  dct_blocks = []

  print(f"Applying DCT to {len(HL_blocks)} blocks...")

  for i, block in enumerate(HL_blocks):
    dct_block = dct_2d(block)
    dct_blocks.append(dct_block)

    # progress indicator
    if (i + 1) % 1000 == 0:
      print(f"  Processed {i + 1}/{len(HL_blocks)} blocks")

  return dct_blocks

def apply_idct(dct_blocks):
  spatial_blocks = []

  print(f"Applying IDCT to {len(dct_blocks)} blocks...")

  for i, dct_block in enumerate(dct_blocks):
    spatial_block = idct_2d(dct_block)
    spatial_blocks.append(spatial_block)

    # progress indicator
    if (i + 1) % 1000 == 0:
      print(f"  Processed {i + 1}/{len(dct_blocks)} blocks")

  return spatial_blocks

In [ ]:
MID_FREQ_MASK = None
MID_FREQ_POS = None

def extract_mid_coeffs(dct_blocks, return_pos=False):
  global MID_FREQ_MASK, MID_FREQ_POS

  if MID_FREQ_MASK is None:
    mid_mask = np.zeros((8, 8), dtype=bool)
    # create a mask for mid-freq coeffs
    for i in range(8):
      for j in range(8):
        freq_sum = i + j
        if 2 <= freq_sum <= 7 and (i, j) != (0, 0):  # mid-freq range
          mid_mask[i, j] = True
    MID_FREQ_MASK = mid_mask
    MID_FREQ_POS = list(zip(*np.where(mid_mask)))

  val = dct_blocks[MID_FREQ_MASK]

  if return_pos:
    return val, MID_FREQ_POS
  else:
    return val

def calculate_scaling_factors(block_chars, base_scaling=8.0, adapt_strength=0.3):
  vars = np.array([char["var"] for char in block_chars])
  ac_energies = np.array([char["ac_energy"] for char in block_chars])

  # normalization characteristics to [0, 1]
  var_norm = (vars - vars.min()) / (vars.max() - vars.min() + 1e-8)
  energy_norm = (ac_energies - ac_energies.min()) / (ac_energies.max() - ac_energies.min() + 1e-8)

  # combine characteristics
  mask_capacity = 0.6 * var_norm + 0.4 * energy_norm

  # calculate adaptive scaling factors
  adapt_factors = adapt_strength * mask_capacity
  scaling_factors = base_scaling * (1.0 + adapt_factors)

  return scaling_factors

In [ ]:
import numpy as np

MID_FREQ_MASK = None
MID_FREQ_POS = None

def extract_mid_coeffs(dct_blocks, return_pos=False):
    global MID_FREQ_MASK, MID_FREQ_POS

    if MID_FREQ_MASK is None:
        mid_mask = np.zeros((8, 8), dtype=bool)
        # create a mask for mid-freq coeffs
        for i in range(8):
            for j in range(8):
                freq_sum = i + j
                if 2 <= freq_sum <= 7 and (i, j) != (0, 0):  # mid-freq range
                    mid_mask[i, j] = True
        MID_FREQ_MASK = mid_mask
        MID_FREQ_POS = list(zip(*np.where(mid_mask)))

    val = dct_blocks[MID_FREQ_MASK]

    if return_pos:
        return val, MID_FREQ_POS
    else:
        return val


def calculate_scaling_factors(block_chars, base_scaling=8.0, adapt_strength=0.3):
    vars = np.array([char["var"] for char in block_chars])
    ac_energies = np.array([char["ac_energy"] for char in block_chars])

    # normalization characteristics to [0, 1]
    var_norm = (vars - vars.min()) / (vars.max() - vars.min() + 1e-8)
    energy_norm = (ac_energies - ac_energies.min()) / (ac_energies.max() - ac_energies.min() + 1e-8)

    # combine characteristics
    mask_capacity = 0.6 * var_norm + 0.4 * energy_norm

    # calculate adaptive scaling factors
    adapt_factors = adapt_strength * mask_capacity
    scaling_factors = base_scaling * (1.0 + adapt_factors)

    return scaling_factors

def select_blocks(dct_blocks, block_pos, wm_len):
    bits_per_block = 1
    var_thres_percent = 70

    blocks_needed = wm_len // bits_per_block
    energy_analysis, all_ac_energies, all_variances = analyze_image_energy(dct_blocks)

    print("\n" + "="*60)
    print("IMAGE ENERGY ANALYSIS")
    print("="*60)
    print(f"\nAC Energy Statistics:")
    print(f"  Total AC Energy:     {energy_analysis['total_ac_energy']:,.2f}")
    print(f"  Mean AC Energy:      {energy_analysis['mean_ac_energy']:,.2f}")
    print(f"  Std AC Energy:       {energy_analysis['ac_energy_std']:,.2f}")
    print(f"  Min AC Energy:       {energy_analysis['ac_energy_min']:,.2f}")
    print(f"  Max AC Energy:       {energy_analysis['ac_energy_max']:,.2f}")
    print(f"  Median AC Energy:    {energy_analysis['ac_energy_median']:,.2f}")

    print(f"\nAC Energy Percentiles:")
    for percentile, value in energy_analysis['ac_energy_percentiles'].items():
        print(f"  {percentile}: {value:,.2f}")

    print(f"\nVariance Statistics:")
    print(f"  Total Variance:      {energy_analysis['total_variance']:,.2f}")
    print(f"  Mean Variance:       {energy_analysis['mean_variance']:,.2f}")
    print(f"  Std Variance:        {energy_analysis['variance_std']:,.2f}")
    print(f"  Min Variance:        {energy_analysis['variance_min']:,.2f}")
    print(f"  Max Variance:        {energy_analysis['variance_max']:,.2f}")
    print(f"  Median Variance:     {energy_analysis['variance_median']:,.2f}")

    print(f"\nVariance Percentiles:")
    for percentile, value in energy_analysis['variance_percentiles'].items():
        print(f"  {percentile}: {value:,.2f}")

    block_vars = []
    block_chars = []

    print("\n" + "="*60)
    print("CALCULATING BLOCK CHARACTERISTICS")
    print("="*60)

    for i, dct_block in enumerate(dct_blocks):
        # calculate var using mid-freq coeffs
        mid_coeffs_val = extract_mid_coeffs(dct_block)
        var = np.var(mid_coeffs_val)
        block_vars.append(var)

        # calculate char for each block
        ac_energy = np.sum(np.abs(dct_block[1:, :])) + np.sum(np.abs(dct_block[0, 1:]))
        max_coeff = np.max(np.abs(dct_block))

        block_chars.append({
            "var": var,
            "ac_energy": ac_energy,
            "max_coeff": max_coeff,
            "block_idx": i
        })

        if (i + 1) % 1000 == 0:
            print(f"  Processed {i + 1}/{len(dct_blocks)} blocks")

    block_vars = np.array(block_vars)

    print("\n" + "="*60)
    print("SELECTING HIGH VARIANCE BLOCKS")
    print("="*60)

    # select high variance blocks
    var_thres = np.percentile(block_vars, var_thres_percent)
    high_var_idx = np.where(block_vars >= var_thres)[0]

    print(f"\nVariance threshold ({var_thres_percent}th percentile): {var_thres:.4f}")
    print(f"Blocks above threshold: {len(high_var_idx)}")
    print(f"Blocks needed: {blocks_needed}")

    if len(high_var_idx) < blocks_needed:
        print(f"\n⚠️  Warning: Not enough high-variance blocks. Lowering threshold...")
        adjusted = False
        for percent in range(var_thres_percent-5, 0, -5):
            var_thres = np.percentile(block_vars, percent)
            high_var_idx = np.where(block_vars >= var_thres)[0]
            print(f"  Trying {percent}th percentile: threshold={var_thres:.4f}, blocks={len(high_var_idx)}")

            if len(high_var_idx) >= blocks_needed:
                adjusted = True
                print(f"  ✓ Found enough blocks at {percent}th percentile")
                break

        if not adjusted:
            print(f"  - Reached minimum threshold; using all available high-variance blocks")

    if len(high_var_idx) > blocks_needed:
        # sort by var and take the top blocks_needed
        sorted_idx = high_var_idx[np.argsort(block_vars[high_var_idx])[::-1]]
        selected_idx = sorted_idx[:blocks_needed]
    else:
        selected_idx = high_var_idx

    print(f"\n✓ Final selected blocks: {len(selected_idx)}")

    print("\n" + "="*60)
    print("SELECTED BLOCKS ANALYSIS")
    print("="*60)

    # extract selected blocks
    selected_blocks = [dct_blocks[i] for i in selected_idx]
    selected_pos = [block_pos[i] for i in selected_idx]
    selected_chars = [block_chars[i] for i in selected_idx]

    # Analyze selected blocks' energy
    selected_ac_energies = np.array([char['ac_energy'] for char in selected_chars])
    selected_variances = np.array([char['var'] for char in selected_chars])
    selected_max_coeffs = np.array([char['max_coeff'] for char in selected_chars])

    print(f"\nSelected Blocks AC Energy:")
    print(f"  Mean:     {selected_ac_energies.mean():,.2f}")
    print(f"  Std:      {selected_ac_energies.std():,.2f}")
    print(f"  Min:      {selected_ac_energies.min():,.2f}")
    print(f"  Max:      {selected_ac_energies.max():,.2f}")
    print(f"  Median:   {np.median(selected_ac_energies):,.2f}")

    print(f"\nSelected Blocks Variance:")
    print(f"  Mean:     {selected_variances.mean():,.2f}")
    print(f"  Std:      {selected_variances.std():,.2f}")
    print(f"  Min:      {selected_variances.min():,.2f}")
    print(f"  Max:      {selected_variances.max():,.2f}")
    print(f"  Median:   {np.median(selected_variances):,.2f}")

    print(f"\nSelected Blocks Max Coefficient:")
    print(f"  Mean:     {selected_max_coeffs.mean():,.2f}")
    print(f"  Std:      {selected_max_coeffs.std():,.2f}")
    print(f"  Min:      {selected_max_coeffs.min():,.2f}")
    print(f"  Max:      {selected_max_coeffs.max():,.2f}")

    print("\n" + "="*60)
    print("CALCULATING ADAPTIVE SCALING FACTORS")
    print("="*60)

    scaling_factors = calculate_scaling_factors(selected_chars)

    print(f"\n📊 Scaling Factor Statistics:")
    print(f"  Mean:     {np.mean(scaling_factors):.4f}")
    print(f"  Std:      {np.std(scaling_factors):.4f}")
    print(f"  Min:      {np.min(scaling_factors):.4f}")
    print(f"  Max:      {np.max(scaling_factors):.4f}")
    print(f"  Median:   {np.median(scaling_factors):.4f}")

    selection_stats = {
        # Overall image analysis
        'image_energy_analysis': energy_analysis,

        # Block selection info
        'total_blocks': len(dct_blocks),
        'blocks_needed': blocks_needed,
        'blocks_selected': len(selected_idx),
        'variance_threshold': var_thres,
        'variance_threshold_percentile': var_thres_percent,

        # Selected blocks statistics
        'selected_variance_stats': {
            'mean': selected_variances.mean(),
            'std': selected_variances.std(),
            'min': selected_variances.min(),
            'max': selected_variances.max(),
            'median': np.median(selected_variances),
        },
        'selected_ac_energy_stats': {
            'mean': selected_ac_energies.mean(),
            'std': selected_ac_energies.std(),
            'min': selected_ac_energies.min(),
            'max': selected_ac_energies.max(),
            'median': np.median(selected_ac_energies),
        },
        'selected_max_coeff_stats': {
            'mean': selected_max_coeffs.mean(),
            'std': selected_max_coeffs.std(),
            'min': selected_max_coeffs.min(),
            'max': selected_max_coeffs.max(),
        },
        'scaling_factor_stats': {
            'mean': np.mean(scaling_factors),
            'std': np.std(scaling_factors),
            'min': np.min(scaling_factors),
            'max': np.max(scaling_factors),
            'median': np.median(scaling_factors),
        }
    }

    print("\n" + "="*60)
    print("SELECTION COMPLETE")
    print("="*60)

    return selected_blocks, selected_pos, selected_idx, scaling_factors

## Compress Watermark

In [ ]:
def compress_wm(wm_array):
  if wm_array.dtype != np.uint8:
    wm_array = wm_array.astype(np.uint8)
  return zlib.compress(wm_array.tobytes(), level=9)

def decompress_wm(compressed_bytes, ori_shape):
  decompressed = zlib.decompress(compressed_bytes)
  decompressed_wm = np.frombuffer(decompressed, dtype=np.uint8).reshape(ori_shape)
  return decompressed_wm

## Scrambling Process

In [ ]:
@numba.njit
def logistic_map(x0, r, n):
  if not 0 < x0 < 1:
    raise ValueError('x0 must be between 0 and 1 for the logistic map')
  if not 3.57 <= r <= 4.0:
    raise ValueError('r should be between 3.57 and 4 for chaotic behavior')

  values = np.zeros(n)
  values[0] = x0
  for i in range(1, n):
    values[i] = r * values[i - 1] * (1 - values[i - 1])
  return values

def scramble_wm(data, r=None, x0=None):
  if x0 is None:
    x0 = secrets.randbelow(9000) / 10000  # 0.0001 ≤ x0 ≤ 0.8999
  if r is None:
    r = 3.7 + secrets.randbelow(20) / 100 # 3.7 ≤ r ≤ 3.9

  n = len(data)

  # generate chaotic sequence and scrambling indices
  sequence = logistic_map(x0, r, n)

  data_array = np.frombuffer(data, dtype=np.uint8)
  sorted_idx = np.argsort(sequence)
  scrambled_wm = data_array[sorted_idx]

  params = {"x0": float(x0), "r": float(r)}
  with open("params.json", "w") as f:
    json.dump(params, f)

  return scrambled_wm.tobytes(), params

def descramble_wm(scrambled_wm, params):
  n = len(scrambled_wm)

  r = params["r"]
  x0 = params["x0"]

  # generate same chaotic sequence
  sequence = logistic_map(x0, r, n)

  scrambled_array = np.frombuffer(scrambled_wm, dtype=np.uint8)
  sorted_idx = np.argsort(sequence)
  inv_idx = np.argsort(sorted_idx)

  descrambled_wm = scrambled_array[inv_idx].tobytes()

  return descrambled_wm

## Add Error Correction

In [ ]:
def encode_error(data, ecc_symbols=128):
  try:
    # create RS codec
    codec = RSCodec(ecc_symbols)

    # encode with error correct
    encoded_data = codec.encode(data)

    print(f"Error correction added:")
    print(f"  - Original data length: {len(data)} bytes")
    print(f"  - Encoded data length: {len(encoded_data)} bytes")
    print(f"  - Error correction symbols: {ecc_symbols}")
    print(f"  - Can correct up to {ecc_symbols//2} errors")

    return encoded_data, codec

  except Exception as e:
    print(f"❌ Error adding error correction: {str(e)}")
    return data, None

def decode_error(encoded_data, codec):
  try:
    # decode with error correct
    result = codec.decode(encoded_data)
    if isinstance(result, tuple) and len(result) == 3:
      decoded_data = result[0]
      correction_made = 0
    else:
      decoded_data = result
      correction_made = 0

    if correction_made > 0:
      print(f"✅ Error correction successful:")
      print(f"  - Corrections made: {correction_made}")
      print(f"  - Recovered data length: {len(decoded_data)} bytes")
    else:
      print("✅ No errors detected in data")

    return decoded_data, correction_made

  except ReedSolomonError as e:
    print(f"❌ Error correction failed: {e}")
    print("  - Too many errors to correct")
    return None, -1

  except Exception as e:
    print(f"❌ Error correcting data: {e}")
    return None, -1

## Encryption

### AES

In [ ]:
# AES constant
S_BOX = [
  0x63, 0x7c, 0x77, 0x7b, 0xf2, 0x6b, 0x6f, 0xc5, 0x30, 0x01, 0x67, 0x2b, 0xfe, 0xd7, 0xab, 0x76,
  0xca, 0x82, 0xc9, 0x7d, 0xfa, 0x59, 0x47, 0xf0, 0xad, 0xd4, 0xa2, 0xaf, 0x9c, 0xa4, 0x72, 0xc0,
  0xb7, 0xfd, 0x93, 0x26, 0x36, 0x3f, 0xf7, 0xcc, 0x34, 0xa5, 0xe5, 0xf1, 0x71, 0xd8, 0x31, 0x15,
  0x04, 0xc7, 0x23, 0xc3, 0x18, 0x96, 0x05, 0x9a, 0x07, 0x12, 0x80, 0xe2, 0xeb, 0x27, 0xb2, 0x75,
  0x09, 0x83, 0x2c, 0x1a, 0x1b, 0x6e, 0x5a, 0xa0, 0x52, 0x3b, 0xd6, 0xb3, 0x29, 0xe3, 0x2f, 0x84,
  0x53, 0xd1, 0x00, 0xed, 0x20, 0xfc, 0xb1, 0x5b, 0x6a, 0xcb, 0xbe, 0x39, 0x4a, 0x4c, 0x58, 0xcf,
  0xd0, 0xef, 0xaa, 0xfb, 0x43, 0x4d, 0x33, 0x85, 0x45, 0xf9, 0x02, 0x7f, 0x50, 0x3c, 0x9f, 0xa8,
  0x51, 0xa3, 0x40, 0x8f, 0x92, 0x9d, 0x38, 0xf5, 0xbc, 0xb6, 0xda, 0x21, 0x10, 0xff, 0xf3, 0xd2,
  0xcd, 0x0c, 0x13, 0xec, 0x5f, 0x97, 0x44, 0x17, 0xc4, 0xa7, 0x7e, 0x3d, 0x64, 0x5d, 0x19, 0x73,
  0x60, 0x81, 0x4f, 0xdc, 0x22, 0x2a, 0x90, 0x88, 0x46, 0xee, 0xb8, 0x14, 0xde, 0x5e, 0x0b, 0xdb,
  0xe0, 0x32, 0x3a, 0x0a, 0x49, 0x06, 0x24, 0x5c, 0xc2, 0xd3, 0xac, 0x62, 0x91, 0x95, 0xe4, 0x79,
  0xe7, 0xc8, 0x37, 0x6d, 0x8d, 0xd5, 0x4e, 0xa9, 0x6c, 0x56, 0xf4, 0xea, 0x65, 0x7a, 0xae, 0x08,
  0xba, 0x78, 0x25, 0x2e, 0x1c, 0xa6, 0xb4, 0xc6, 0xe8, 0xdd, 0x74, 0x1f, 0x4b, 0xbd, 0x8b, 0x8a,
  0x70, 0x3e, 0xb5, 0x66, 0x48, 0x03, 0xf6, 0x0e, 0x61, 0x35, 0x57, 0xb9, 0x86, 0xc1, 0x1d, 0x9e,
  0xe1, 0xf8, 0x98, 0x11, 0x69, 0xd9, 0x8e, 0x94, 0x9b, 0x1e, 0x87, 0xe9, 0xce, 0x55, 0x28, 0xdf,
  0x8c, 0xa1, 0x89, 0x0d, 0xbf, 0xe6, 0x42, 0x68, 0x41, 0x99, 0x2d, 0x0f, 0xb0, 0x54, 0xbb, 0x16
]

INV_S_BOX = [
  0x52, 0x09, 0x6a, 0xd5, 0x30, 0x36, 0xa5, 0x38, 0xbf, 0x40, 0xa3, 0x9e, 0x81, 0xf3, 0xd7, 0xfb,
  0x7c, 0xe3, 0x39, 0x82, 0x9b, 0x2f, 0xff, 0x87, 0x34, 0x8e, 0x43, 0x44, 0xc4, 0xde, 0xe9, 0xcb,
  0x54, 0x7b, 0x94, 0x32, 0xa6, 0xc2, 0x23, 0x3d, 0xee, 0x4c, 0x95, 0x0b, 0x42, 0xfa, 0xc3, 0x4e,
  0x08, 0x2e, 0xa1, 0x66, 0x28, 0xd9, 0x24, 0xb2, 0x76, 0x5b, 0xa2, 0x49, 0x6d, 0x8b, 0xd1, 0x25,
  0x72, 0xf8, 0xf6, 0x64, 0x86, 0x68, 0x98, 0x16, 0xd4, 0xa4, 0x5c, 0xcc, 0x5d, 0x65, 0xb6, 0x92,
  0x6c, 0x70, 0x48, 0x50, 0xfd, 0xed, 0xb9, 0xda, 0x5e, 0x15, 0x46, 0x57, 0xa7, 0x8d, 0x9d, 0x84,
  0x90, 0xd8, 0xab, 0x00, 0x8c, 0xbc, 0xd3, 0x0a, 0xf7, 0xe4, 0x58, 0x05, 0xb8, 0xb3, 0x45, 0x06,
  0xd0, 0x2c, 0x1e, 0x8f, 0xca, 0x3f, 0x0f, 0x02, 0xc1, 0xaf, 0xbd, 0x03, 0x01, 0x13, 0x8a, 0x6b,
  0x3a, 0x91, 0x11, 0x41, 0x4f, 0x67, 0xdc, 0xea, 0x97, 0xf2, 0xcf, 0xce, 0xf0, 0xb4, 0xe6, 0x73,
  0x96, 0xac, 0x74, 0x22, 0xe7, 0xad, 0x35, 0x85, 0xe2, 0xf9, 0x37, 0xe8, 0x1c, 0x75, 0xdf, 0x6e,
  0x47, 0xf1, 0x1a, 0x71, 0x1d, 0x29, 0xc5, 0x89, 0x6f, 0xb7, 0x62, 0x0e, 0xaa, 0x18, 0xbe, 0x1b,
  0xfc, 0x56, 0x3e, 0x4b, 0xc6, 0xd2, 0x79, 0x20, 0x9a, 0xdb, 0xc0, 0xfe, 0x78, 0xcd, 0x5a, 0xf4,
  0x1f, 0xdd, 0xa8, 0x33, 0x88, 0x07, 0xc7, 0x31, 0xb1, 0x12, 0x10, 0x59, 0x27, 0x80, 0xec, 0x5f,
  0x60, 0x51, 0x7f, 0xa9, 0x19, 0xb5, 0x4a, 0x0d, 0x2d, 0xe5, 0x7a, 0x9f, 0x93, 0xc9, 0x9c, 0xef,
  0xa0, 0xe0, 0x3b, 0x4d, 0xae, 0x2a, 0xf5, 0xb0, 0xc8, 0xeb, 0xbb, 0x3c, 0x83, 0x53, 0x99, 0x61,
  0x17, 0x2b, 0x04, 0x7e, 0xba, 0x77, 0xd6, 0x26, 0xe1, 0x69, 0x14, 0x63, 0x55, 0x21, 0x0c, 0x7d
]

RCON = [0x01, 0x02, 0x04, 0x08, 0x10, 0x20, 0x40, 0x80, 0x1B, 0x36]

def key_expansion(key, nb=4, nr=10):
  nk = len(key) // 4

  key_words = [0] * nk
  for i in range(nk):
    key_words[i] = (key[4*i] << 24) | (key[4*i+1] << 16) | (key[4*i+2] << 8) | key[4*i+3]

  # expand key
  expanded_key = [0] * (nb * (nr + 1))
  for i in range(nk):
    expanded_key[i] = key_words[i]

  for i in range(nk, nb * (nr + 1)):
    temp = expanded_key[i - 1]

    if i % nk == 0:
      # rotword and subword with RCON
      temp = ((temp << 8) | (temp >> 24)) & 0xFFFFFFFF #rotword

      # subword
      temp = ((S_BOX[(temp >> 24) & 0xFF] << 24) |
              (S_BOX[(temp >> 16) & 0xFF] << 16) |
              (S_BOX[(temp >> 8) & 0xFF] << 8) |
              S_BOX[temp & 0xFF])

      # XOR with RCON
      temp ^= (RCON[(i // nk) - 1] << 24)

    elif nk > 6 and i % nk == 4:
      # subword for AES 256
      temp = ((S_BOX[(temp >> 24) & 0xFF] << 24) |
              (S_BOX[(temp >> 16) & 0xFF] << 16) |
              (S_BOX[(temp >> 8) & 0xFF] << 8) |
              S_BOX[temp & 0xFF])

    expanded_key[i] = expanded_key[i - nk] ^ temp

  return expanded_key

def sub_bytes(state):
  for i in range(4):
    for j in range(4):
      state[i][j] = S_BOX[state[i][j]]
  return state

def inv_sub_bytes(state):
  for i in range(4):
    for j in range(4):
      state[i][j] = INV_S_BOX[state[i][j]]
  return state

def shift_rows(state):
  state[1] = np.roll(state[1], -1)
  state[2] = np.roll(state[2], -2)
  state[3] = np.roll(state[3], -3)
  return state

def inv_shift_rows(state):
  # shift right
  state[1] = np.roll(state[1], 1)
  state[2] = np.roll(state[2], 2)
  state[3] = np.roll(state[3], 3)
  return state

def gmul(a, b):
    p = 0
    for _ in range(8):
      if b & 1:
        p ^= a
      hi_bit_set = a & 0x80
      a <<= 1
      if hi_bit_set:
        a ^= 0x1B
      b >>= 1
    return p & 0xFF

def mix_columns(state):
  new_state = [[0 for _ in range(4)] for _ in range(4)]

  for j in range(4):
    s0 = state[0][j]
    s1 = state[1][j]
    s2 = state[2][j]
    s3 = state[3][j]

    new_state[0][j] = gmul(s0, 2) ^ gmul(s1, 3) ^ s2 ^ s3
    new_state[1][j] = s0 ^ gmul(s1, 2) ^ gmul(s2, 3) ^ s3
    new_state[2][j] = s0 ^ s1 ^ gmul(s2, 2) ^ gmul(s3, 3)
    new_state[3][j] = gmul(s0, 3) ^ s1 ^ s2 ^ gmul(s3, 2)

  return new_state

def inv_mix_columns(state):
  new_state = [[0 for _ in range(4)] for _ in range(4)]
  for i in range(4):
    col = [state[0][i], state[1][i], state[2][i], state[3][i]]
    new_col = [
        gmul(0x0e, col[0]) ^ gmul(0x0b, col[1]) ^
        gmul(0x0d, col[2]) ^ gmul(0x09, col[3]),
        gmul(0x09, col[0]) ^ gmul(0x0e, col[1]) ^
        gmul(0x0b, col[2]) ^ gmul(0x0d, col[3]),
        gmul(0x0d, col[0]) ^ gmul(0x09, col[1]) ^
        gmul(0x0e, col[2]) ^ gmul(0x0b, col[3]),
        gmul(0x0b, col[0]) ^ gmul(0x0d, col[1]) ^
        gmul(0x09, col[2]) ^ gmul(0x0e, col[3])
    ]
    for j in range(4):
      new_state[j][i] = new_col[j] & 0xFF

  return new_state

def add_round_key(state, round_key):
  for i in range(4):
    for j in range(4):
      key_byte = (round_key[j] >> (24 - 8 * i)) & 0xFF
      state[i][j] ^= key_byte
  return state

def aes_encrypt_block(block, expanded_key, nr=10):
  # convert block to 4*4 state matrix
  state = np.zeros((4, 4), dtype=np.uint8)
  for i in range(4):
    for j in range(4):
      state[j][i] = block[i*4 + j]

  # initialize round_key addition
  state = add_round_key(state, expanded_key[:4])

  # main roound
  for round in range(1, nr):
    state = sub_bytes(state)
    state = shift_rows(state)
    state = mix_columns(state)
    state = add_round_key(state, expanded_key[4*round:4*(round+1)])

  # final round
  state = sub_bytes(state)
  state = shift_rows(state)
  state = add_round_key(state, expanded_key[4*nr:4*(nr+1)])

  # convert array to block
  result = bytearray(16)
  for i in range(4):
    for j in range(4):
      result[i*4 + j] = state[j][i]

  return result

def aes_decrypt_block(block, expanded_key, INV_S_BOX, nr=10):
  # convert block to array
  state = np.zeros((4, 4), dtype=np.uint8)

  for i in range(4):
    for j in range(4):
      state[j][i] = block[i*4 + j]

  # initial round
  state = add_round_key(state, expanded_key[4*nr:4*(nr+1)])

  # main rounds
  for round in range(nr-1, 0, -1):
    state = inv_shift_rows(state)
    state = inv_sub_bytes(state)
    state = add_round_key(state, expanded_key[round*4:(round+1)*4])
    state = inv_mix_columns(state)

  # final round
  state = inv_shift_rows(state)
  state = inv_sub_bytes(state)
  state = add_round_key(state, expanded_key[:4])

  # convert state to block
  result = bytearray(16)
  for i in range(4):
    for j in range(4):
      result[i*4 + j] = state[j][i]

  return result

def aes_encrypt_cbc(data, key, iv):
  if len(key) != 16:
    raise ValueError('Key must be 16 bytes long')

  expanded_key = key_expansion(key)

  # padding using PKCS#7
  pad_len = 16 - (len(data) % 16)
  if pad_len == 0:
    pad_len = 16
  padded_data = data + bytes([pad_len] * pad_len)

  # split data into blocks
  blocks = [padded_data[i:i+16] for i in range(0, len(padded_data), 16)]

  # encrypt blocks in CBC
  encrypted_data = bytearray()
  prev_block = iv

  for block in blocks:
    xor_block = bytearray([block[i] ^ prev_block[i] for i in range(16)])

    # encrypt block
    encrypted_block = aes_encrypt_block(xor_block, expanded_key, nr=10)

    # append to result
    encrypted_data.extend(encrypted_block)

    # update prev block
    prev_block = encrypted_block

  print(f"Pad length: {pad_len}")
  print(f"Original data length: {len(data)}")
  print(f"Encrypted data length: {len(encrypted_data)}")

  return encrypted_data

def aes_decrypt_cbc(encrypted_data, key, iv):
    if len(key) != 16:
        raise ValueError('Key must be 16 bytes long')

    expanded_key = key_expansion(key)

    # split data into blocks
    blocks = [encrypted_data[i:i+16] for i in range(0, len(encrypted_data), 16)]

    # decrypt blocks
    decrypted_data = bytearray()
    prev_block = iv

    for block in blocks:
        # decrypt the block
        decrypted_block = aes_decrypt_block(block, expanded_key, INV_S_BOX, nr=10)

        xor_block = bytearray([decrypted_block[i] ^ prev_block[i] for i in range(16)])

        # append to result
        decrypted_data.extend(xor_block)

        # update prev block
        prev_block = block

    # Padding validation - raise on failure
    if len(decrypted_data) == 0:
        raise ValueError("Decrypted data is empty")

    pad_len = decrypted_data[-1]
    if not (1 <= pad_len <= 16):
        raise ValueError("Invalid padding length: pad_len is out of range")
    if not all(b == pad_len for b in decrypted_data[-pad_len:]):
        raise ValueError("Invalid padding: padding bytes do not match pad_len")

    decrypted_data = decrypted_data[:-pad_len]
    return bytes(decrypted_data)

In [ ]:
def bytes_to_int(data):
  return int.from_bytes(data, byteorder='big')

def int_to_bytes(integer, key_length):
  return integer.to_bytes(key_length, byteorder='big')

### Watermark Encryption

In [ ]:
def encrypt_scrambled_wm(scrambled_wm):
  # generate random AES-128 key (16 bytes)
  aes_key = os.urandom(16)
  iv = os.urandom(16)

  encrypted_wm = aes_encrypt_cbc(scrambled_wm, aes_key, iv)

  # save encrypted data
  result = {
      "encrypted_wm": list(encrypted_wm),
      "iv": list(iv),
      "aes_key": list(aes_key)
  }

  return result

def decrypt_encrypted_wm(encrypted_data):
  encrypted_wm = bytes(encrypted_data["encrypted_wm"])
  aes_key = bytes(encrypted_data["aes_key"])
  iv = bytes(encrypted_data["iv"])

  # decrypt wm with AES
  decrypted_wm = aes_decrypt_cbc(encrypted_wm, aes_key, iv)

  return decrypted_wm

In [ ]:
def prepare_data_for_embed(encrypted_data):
  encrypted_wm = np.array(encrypted_data["encrypted_wm"], dtype=np.uint8)
  # convert to bit
  encrypted_wm = np.unpackbits(np.frombuffer(encrypted_wm, dtype=np.uint8))

  wm_len = len(encrypted_wm)

  return encrypted_wm, wm_len

## Embedding Process

In [ ]:
def qim_embed(coeff: float, bit: int, delta: float) -> float:
  eps = 1e-10
  q = np.round((coeff + eps) / delta)

  if bit == 0:
    q_new = 2 * np.round(q / 2)
  else:
    q_new = 2 * np.round((q - 1) / 2) + 1

  embed_coeff = q_new * delta

  return embed_coeff

def qim_extract(coeff, delta):
  q = np.round(coeff / delta)

  return 0 if int(q) % 2 == 0 else 1

In [ ]:
def embed_bits(selected_blocks, scaling_factors, data_bits):
  bits_per_block = 1
  blocks_needed = len(data_bits)

  if len(selected_blocks) < blocks_needed:
    raise ValueError(f"Not enough selected blocks. Need {blocks_needed}, got {len(selected_blocks)}")

  if len(scaling_factors) != len(selected_blocks):
    raise ValueError(f"Scaling factors and selected blocks must be the same length. Got {len(scaling_factors)} and {len(selected_blocks)}")

  modif_blocks = []
  all_embed_positions = []

  embed_stats = {
    'total_modifications': 0,
    'avg_modification': 0.0,
    'max_modification': 0.0
  }

  _, sample_pos = extract_mid_coeffs(selected_blocks[0], return_pos=True)
  pos_per_block = len(sample_pos)

  print(f"\nEmbedding watermark using QIM...")
  print(f"  - Embedding 1 bit per block to {pos_per_block} positions")
  print(f"  - Total bits to embed: {len(data_bits)}")
  print(f"  - Blocks to use: {blocks_needed}")
  print(f"  - Bits per block: {bits_per_block}")

  # embed bits into blocks
  for block_idx in range(blocks_needed):
    dct_block = selected_blocks[block_idx]
    delta = scaling_factors[block_idx]
    bit_val = int(data_bits[block_idx])

    mid_coeffs, embed_pos = extract_mid_coeffs(dct_block, return_pos=True)

    if len(embed_pos) == 0:
      raise ValueError(f"No mid-frequency positions found in block {block_idx}")

    modif_block = dct_block.copy()
    embed_positions = []

    # embed bit to multiple positions
    for pos_row, pos_col in embed_pos:
      # get ori coeff
      ori_coeff = dct_block[pos_row, pos_col]
      quantized_coeff = qim_embed(ori_coeff, bit_val, delta)
      modif_block[pos_row, pos_col] = quantized_coeff

      embed_positions.append((pos_row, pos_col))

      modif = abs(quantized_coeff - ori_coeff)
      embed_stats['total_modifications'] += 1
      embed_stats['max_modification'] = max(embed_stats['max_modification'], modif)
      embed_stats['avg_modification'] += modif

    modif_blocks.append(modif_block)
    all_embed_positions.append(embed_positions)
    # progress report
    if (block_idx + 1) % 500 == 0:
      print(f"  - Embedded {block_idx + 1}/{blocks_needed} blocks")

  if blocks_needed > 0:
    embed_stats['avg_modification'] /= blocks_needed

  print(f"  - Embedding completed!")
  print(f"  - Total coefficient modifications: {embed_stats['total_modifications']}")
  print(f"  - Average modification per block: {embed_stats['avg_modification']:.4f}")
  print(f"  - Maximum modification: {embed_stats['max_modification']:.4f}")

  # verify embedding capacity
  actual_embed_bits = blocks_needed * bits_per_block
  print(f"  - Bits embedded: {actual_embed_bits}/{len(data_bits)}")

  if actual_embed_bits < len(data_bits):
    print(f"  - Warning: Not all bits embedded due to insufficient blocks!")

  return modif_blocks, all_embed_positions

## Image Reconstruction

In [ ]:
def recons_watermarked_img(subbands, dct_blocks, modif_blocks, selected_idx,
                           block_pos, HL_shape, ori_shape, levels=1):
  print(f"\n=== WATERMARKED IMAGE RECONSTRUCTION ===")

  # recons HL subband
  reconstructed_HL = recons_HL(dct_blocks, modif_blocks, selected_idx,
                        block_pos, HL_shape)

  # update subband dict
  updated_subbands = subbands.copy()
  updated_subbands["HL1"] = reconstructed_HL
  print(f"  - Updated HL1 subband with watermarked content")

  # apply idwt to recons image
  print(f"  - Applying {levels}-level IDWT...")
  watermarked_img = apply_idwt(updated_subbands, ori_shape, levels).astype(np.float32)
  watermarked_img = np.clip(watermarked_img, 0, 255)

  print(f"  - Final watermarked image shape: {watermarked_img.shape}")
  cv2.imwrite('watermarked.png', watermarked_img.astype(np.uint8))

  print(f"=== RECONSTRUCTION COMPLETED ===\n")

  return watermarked_img

## Embedding Pipeline

In [ ]:
def calculate_psnr(original, watermarked):
    if original.shape != watermarked.shape:
        raise ValueError(f"Image shapes don't match: {original.shape} vs {watermarked.shape}")

    # Convert to float32 for calculation
    original_float = original.astype(np.float32)
    watermarked_float = watermarked.astype(np.float32)

    # Calculate MSE
    mse = np.mean((original_float - watermarked_float) ** 2)

    # Handle perfect match
    if mse == 0:
        return 100.0  # or np.inf

    # Calculate PSNR
    max_pixel = 255.0
    psnr = 20 * np.log10(max_pixel / np.sqrt(mse))

    return psnr


def calculate_ssim(original, watermarked):
    if original.shape != watermarked.shape:
        raise ValueError(f"Image shapes don't match: {original.shape} vs {watermarked.shape}")

    # Handle grayscale vs color images
    if len(original.shape) == 3:
        # Color image - calculate SSIM for each channel and average
        ssim_value = structural_similarity(original, watermarked, channel_axis=2, data_range=255)
    else:
        # Grayscale image
        ssim_value = structural_similarity(original, watermarked, data_range=255)

    return ssim_value

In [ ]:
def embed_pipeline(host_img, wm, levels=1):
  print("=== WATERMARK EMBEDDING PIPELINE ===\n")

  # Host Image Processing
  print("Host Image Processing...")
  subbands = apply_dwt(host_img, levels=levels)
  HL_blocks, HL_shape, block_pos = split_HL(subbands)
  dct_blocks = apply_dct(HL_blocks)

  # Watermark Preprocessing
  print("\nWatermark Pre-processing...")
  compressed_wm = compress_wm(wm)
  scrambled_wm, params = scramble_wm(compressed_wm)
  ecc_symbols = 96
  protected_wm, wm_ecc_codec = encode_error(scrambled_wm, ecc_symbols=ecc_symbols)
  data_bits = np.unpackbits(np.frombuffer(protected_wm, dtype=np.uint8))
  wm_len = len(data_bits)

  print(f"Payload: {len(compressed_wm)} → {len(scrambled_wm)} → {len(protected_wm)} bytes (+{ecc_symbols} ECC)")
  print(f"→ Embedding {wm_len} bits (need ~{wm_len} blocks)")

  selected_blocks, selected_pos, selected_idx, scaling_factors = select_blocks(dct_blocks, block_pos, wm_len)

  # Embedding Process
  print("\nWatermark Embedding...")
  modified_blocks, all_embed_positions = embed_bits(selected_blocks, scaling_factors,
                                                data_bits)

  # Image Reconstruction
  print("\nImage Reconstruction...")
  watermarked_image = recons_watermarked_img(subbands, dct_blocks,
                                             modified_blocks, selected_idx,
                                             block_pos, HL_shape, host_img.shape,
                                             levels)

  embedding_info = {
      'scaling_factors': scaling_factors,
      'selected_idx': selected_idx,
      'selected_pos': selected_pos,
      'all_embed_positions': all_embed_positions,
      'HL_shape': HL_shape,
      'params': params,
      'wm_len': wm_len,
      'aes_key': encrypted_data['aes_key'],
      'encrypted_wm': encrypted_data['encrypted_wm'],
      'iv': encrypted_data['iv'],
      'wm_ecc_codec': wm_ecc_codec
  }

  psnr_value, ssim_value = display_image_quality_metrics(host_img, watermarked_image)


  print("=== PIPELINE COMPLETED ===\n")

  return watermarked_image, embedding_info

In [ ]:
def embed_pipeline_colored(host_img, wm, levels=2):
  if len(host_img.shape) == 3:
    print("Color image detected, converting to YCbCr...")

    ycbcr_img = cv2.cvtColor(host_img, cv2.COLOR_BGR2YCrCb)
    y_channel = ycbcr_img[:, :, 0]
    cb_channel = ycbcr_img[:, :, 1]
    cr_channel = ycbcr_img[:, :, 2]

    print("Embedding watermark in Y (luminance) channel...")
    watermarked_y, embedding_info = embed_pipeline(y_channel, wm, levels)

    # reconstruct color image
    watermarked_ycbcr = np.stack([
        watermarked_y.astype(np.uint8),
        cb_channel,
        cr_channel
    ], axis=2)

    watermarked_img = cv2.cvtColor(watermarked_ycbcr, cv2.COLOR_YCrCb2BGR)

  else:
    print("Grayscale image detected, embedding watermark...")
    watermarked_img, embedding_info = embed_pipeline(host_img, wm, levels)

  return watermarked_img, embedding_info

In [ ]:
watermarked_image, embedding_info = embed_pipeline_colored(host_img, wm, levels=1)

In [ ]:
cv2_imshow(watermarked_image)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(cv2.cvtColor(host_img, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original Image")
axes[0].axis("off")

axes[1].imshow(cv2.cvtColor(watermarked_image, cv2.COLOR_BGR2RGB))
axes[1].set_title("Watermarked Image")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## Save Watermarked Image to Drive

In [ ]:
def save_to_drive(watermarked_img, img_idx, folder_path):
  if not os.path.exists(folder_path):
    os.makedirs(folder_path)

  if watermarked_img.dtype != np.uint8:
    watermarked_img = watermarked_img.astype(np.uint8)

  img_name = f"watermarked_img{img_idx}.png"
  img_path = os.path.join(folder_path, img_name)

  if os.path.exists(img_path):
    print(f"ℹ️ The watermarked image already exists in {folder_path} folder.")
    print(f"  Path: {img_path}")
    return False

  try:
    success = cv2.imwrite(img_path, watermarked_img)
    if success:
      print(f"✓ Watermarked image saved successfully: {img_path}")
      if os.path.exists(img_path):
        file_size = os.path.getsize(img_path) / 1024
        print(f"  - File size: {file_size:.2f} KB")
        return True
      else:
        print("⚠️ Warning: cv2.imwrite returned success but file not created")
        return False

  except Exception as e:
    print(f"✗ Error saving watermarked image: {e}")
    return False

  return True

In [ ]:
watermarked_image_dir = "/content/drive/My Drive/Tugas Akhir/Watermarked Image"
save_image2 = save_to_drive(watermarked_image, 1, watermarked_image_dir)

# Extraction Section

In [ ]:
def extract_blocks(dct_blocks, embedding_info):
  selected_idx = embedding_info['selected_idx']
  wm_len = embedding_info['wm_len']

  bits_per_block = 1
  blocks_needed = wm_len // bits_per_block

  print(f"\nExtracting bits from {len(selected_idx)} selected blocks...")
  print(f"Expected bits to extract: {blocks_needed}")

  if len(selected_idx) < blocks_needed:
    raise ValueError(f"Not enough selected blocks. Need {blocks_needed}, got {len(selected_idx)}")

  selected_blocks = [dct_blocks[idx] for idx in selected_idx]

  return selected_blocks

In [ ]:
def extract_bits(selected_blocks, embedding_info):
  scaling_factors = embedding_info['scaling_factors']
  wm_len = embedding_info['wm_len']
  all_embed_positions = embedding_info['all_embed_positions']

  bits_per_block = 1
  blocks_needed = wm_len // bits_per_block

  print(f"\nExtracting watermark...")
  print(f"  - Blocks to process: {min(blocks_needed, len(selected_blocks))}")
  print(f"  - Expected total bits: {wm_len}")

  extracted_bits = []

  for block_idx in range(min(blocks_needed, len(selected_blocks))):
    dct_block = selected_blocks[block_idx]
    delta = scaling_factors[block_idx]
    positions = all_embed_positions[block_idx]

    if len(positions) == 0:
      print(f"Warning: No embeddign positions in block {block_idx}. Skipping")

    bits_this_block = []
    for pos_row, pos_col in positions:
      coeff_val = dct_block[pos_row, pos_col]
      extracted_bit = qim_extract(coeff_val, delta)
      bits_this_block.append(extracted_bit)

    # majority vote
    if bits_this_block:
      ones_count = sum(bits_this_block)
      zeros_count = len(bits_this_block) - ones_count
      final_bit = 1 if ones_count > zeros_count else 0
      extracted_bits.append(final_bit)
    else:
      extracted_bits.append(0) # fallback

    if len(extracted_bits) >= wm_len:
      break

  extracted_bits = np.array(extracted_bits[:wm_len], dtype=np.uint8)
  print(f"  - Extracted {len(extracted_bits)} bits")

  return extracted_bits

def fallback_from_bits(bits):
  if len(bits) == 0:
      return np.zeros((8, 8), dtype=np.uint8)
  bits = np.array(bits).flatten()
  total_needed = 64 * 64  # Fixed size for QR
  padded_bits = np.pad(bits[:total_needed], (0, max(0, total_needed - len(bits))), mode='constant')
  arr = padded_bits.reshape(64, 64) * 255
  return arr.astype(np.uint8)

## Extraction Pipeline

In [ ]:
def extract_pipeline(watermarked_img, embedding_info, levels=1):
  print("=== WATERMARK EXTRACTION ===\n")

  selected_idx = embedding_info['selected_idx']
  selected_pos = embedding_info['selected_pos']
  scaling_factors = embedding_info['scaling_factors']

  success = True
  err_stage, err_msg = None, None
  tampered_wm = None
  extracted_wm = None

  print("\nDWT and Block Preparation")
  subbands = apply_dwt(watermarked_img, levels=levels)
  HL_blocks, HL_shape, block_pos = split_HL(subbands)

  print("\nBlock Selection")

  dct_blocks = apply_dct(HL_blocks)
  selected_blocks = extract_blocks(dct_blocks, embedding_info)

  print(f"  - Recreated {len(selected_blocks)} selected blocks")
  print(f"  - Scaling factors range: [{np.min(scaling_factors):.3f}, {np.max(scaling_factors):.3f}]")

  print("\nBit Extraction")
  extracted_bits = extract_bits(selected_blocks, embedding_info)

  print("\nData Separation and Decryption")
  wm_len = embedding_info['wm_len']
  extracted_wm_bits = extracted_bits[:wm_len]
  print(f"  - Extracted watermark bits: {len(extracted_wm_bits)}")

  extracted_wm_bytes = np.packbits(extracted_wm_bits.reshape(-1, 8),
                                   axis=1).flatten().tobytes()

  # Error Correction for Watermark
  print("\nError Correction for Watermark")
  wm_ecc_codec = embedding_info.get('wm_ecc_codec')
  if wm_ecc_codec is None:
    print("❌ No Reed-Solomon codec found - skipping correction")
    corrected_bytes = extracted_wm_bytes
    corrections = 0
  else:
    try:
      corrected_bytes, corrections = decode_error(extracted_wm_bytes, wm_ecc_codec)
      print(f"Reed-Solomon corrected {corrections} byte errors")

    except Exception as e:
      success = False
      err_stage, err_msg = "Watermark Error Correction", str(e)
      tampered_wm = fallback_from_bits(extracted_wm_bits)

  if success:
    print("\nAES Decryption")
    try:
      # decrypt using AES
      key = embedding_info['aes_key']
      iv = embedding_info['iv']
      decrypted_wm = aes_decrypt_cbc(extracted_wm_bytes, key, iv)

    except Exception as e:
      success = False
      err_stage, err_msg = "AES Decryption", str(e)
      tampered_wm = fallback_from_bits(extracted_wm_bits)
      decrypted_wm = extracted_wm_bits

  if success:
    print("\nDescrambling Watermark")
    try:
      params = embedding_info['params']
      descrambled_wm = descramble_wm(corrected_bytes, params)
      print(f"  - Watermark descrambled successfully")

    except Exception as e:
      success = False
      err_stage, err_msg = "Watermark descrambling", str(e)
      try:
        tampered_wm = np.frombuffer(decrypted_wm, dtype=np.uint8)[:64*64].reshape(64, 64)
      except:
        tampered_wm = fallback_from_bits(extracted_wm_bits)

  if success:
    print("\nDecompress Watermark")
    try:
      # debug
      # Check zlib header - should start with 0x78
      if len(descrambled_wm) >= 2:
        header = descrambled_wm[:2]
        print(f"Zlib header check: {header.hex()}")
        if header[0] == 0x78:
          print("✅ Valid zlib header detected")
        else:
          print(f"❌ Invalid zlib header. Expected 0x78xx, got {header.hex()}")
      # end of debug

      extracted_wm = decompress_wm(descrambled_wm, ori_shape=(64, 64))
      print(f"  - Watermark decompressed successfully")

    except Exception as e:
      success = False
      err_stage, err_msg = "Watermark decompression", str(e)
      # fallback: reshape the bytes to an image
      try:
        tampered_wm = np.frombuffer(descrambled_wm, dtype=np.uint8)[:64*64].reshape(64, 64)
      except:
        tampered_wm = fallback_from_bits(extracted_wm_bits)
  print("\n=== WATERMARK EXTRACTION COMPLETED ===\n")

  return {
      'success': success,
      'extracted_wm': extracted_wm if success else None,
      'tampered_wm': tampered_wm if tampered_wm is not None else fallback_from_bits(extracted_wm),
      'extracted_bits': extracted_bits,
      'err_stage': err_stage,
      'err_msg': err_msg
  }

In [ ]:
def extract_pipeline_colored(watermarked_img, embedding_info, levels=1):
  if len(watermarked_img.shape) == 3:
    ycbcr_img = cv2.cvtColor(watermarked_img, cv2.COLOR_BGR2YCrCb)
    y_channel = ycbcr_img[:, :, 0]

    print("Extracting watermark from Y (luminance) channel...")
    extract_info = extract_pipeline(y_channel, embedding_info, levels)
  else:
    print("Grayscale image detected, extracting watermark...")
    extract_info = extract_pipeline(watermarked_img, embedding_info, levels)

  return extract_info

In [ ]:
extract_info = extract_pipeline_colored(watermarked_image, embedding_info, levels=1)

In [ ]:
cv2_imshow(extract_info['extracted_wm'])

In [ ]:
def save_wm_to_drive(wm_ext, wm_idx, folder_path):
  if not os.path.exists(folder_path):
    os.makedirs(folder_path)

  if wm_ext.dtype != np.uint8:
    wm_ext = wm_ext.astype(np.uint8)

  wm_name = f"wm_ext{wm_idx}.png"
  wm_path = os.path.join(folder_path, wm_name)

  if os.path.exists(wm_path):
    print(f"ℹ️ The extracted watermark already exists in {folder_path} folder.")
    print(f"  Path: {wm_path}")
    return False

  try:
    success = cv2.imwrite(wm_path, wm_ext)
    if success:
      print(f"✓ Extracted watermark saved successfully: {wm_path}")
      if os.path.exists(wm_path):
        file_size = os.path.getsize(wm_path) / 1024
        print(f"  - File size: {file_size:.2f} KB")
        return True
      else:
        print("⚠️ Warning: cv2.imwrite returned success but file not created")
        return False

  except Exception as e:
    print(f"✗ Error saving extracted watermark: {e}")
    return False

  return True

In [ ]:
wm_extracted_dir = "/content/drive/My Drive/Tugas Akhir/Extracted Watermark"
save_wm_to_drive = save_wm_to_drive(test_extract_info['extracted_wm'], 1, wm_extracted_dir)

# Verify Watermark

In [ ]:
def calculate_psnr(original, watermarked):
  mse = np.mean((original.astype(np.float32) - watermarked.astype(np.float32)) ** 2)
  if mse == 0:
    return float('inf')
  max_pixel = 255.0
  psnr = 20 * np.log10(max_pixel / np.sqrt(mse))

  return psnr

def calculate_ssim(original, watermarked):
   return ssim(original, watermarked, channel_axis=2)

def calculate_nc(original, extracted):
    # Convert to float
    orig_f = original.astype(np.float32)
    extr_f = extracted.astype(np.float32)

    # Calculate NC
    numerator = np.sum(orig_f * extr_f)
    denominator = np.sqrt(np.sum(orig_f**2) * np.sum(extr_f**2))

    if denominator == 0:
        return 0.0

    nc = numerator / denominator
    return float(nc)

def calculate_ber(original_bits, extracted_bits):
    if len(original_bits) != len(extracted_bits):
        return 1.0  # Maximum error

    errors = np.sum(original_bits != extracted_bits)
    ber = errors / len(original_bits)
    return float(ber)

def wm_robustness(ori_wm, extracted_wm):
  print("Robustness Assessment")
  metrics = {}

  if ori_wm is not None:
    nc = calculate_nc(ori_wm, extracted_wm)

    ori_bits = np.unpackbits(ori_wm.astype(np.uint8).flatten())
    extracted_bits = np.unpackbits(extracted_wm.astype(np.uint8).flatten())

    if len(ori_bits) == len(extracted_bits):
      ber = calculate_ber(ori_bits, extracted_bits)
    else:
      ber = 1.0

    metrics = {
        'nc': nc,
        'ber': ber
    }

    print(f"  - NC (Normalized Correlation): {nc:.4f}")
    print(f"  - BER (Bit Error Rate): {ber:.4f}")

  else:
    print("  - No original watermark provided for quality assessment")

  return metrics

def wm_imperceptibility(original, watermarked):
  print("\nImperceptibility Assessment")
  metrics = {}

  if original is None or watermarked is None:
    print("  - Original or watermarked image is missing for imperceptibility assessment")
    return metrics

  try:
    psnr_val = calculate_psnr(original, watermarked)
    metrics['psnr'] = psnr
    print(f"  - PSNR (Peak Signal-to-Noise Ratio): {psnr:.3f} dB")

    ssim_val = calculate_ssim(original, watermarked)
    metrics['ssim'] = ssim_val
    print(f"  - SSIM (Structural Similarity Index): {ssim_val:.3f}")

  except Exception as e:
    print(f"  - Error in imperceptibility assessment: {str(e)}")

  return metrics

In [ ]:
test_robustness = wm_robustness(wm, extract_info['extracted_wm'])
test_imperceptibility = wm_imperceptibility(host_img, watermarked_image)

## Uji Robustness and Imperceptibility

In [ ]:
def calculate_ber(original_bits, extracted_bits):
    original_bits = np.array(original_bits, dtype=np.uint8)
    extracted_bits = np.array(extracted_bits, dtype=np.uint8)

    if len(extracted_bits) != len(original_bits):
        print(f"Warning: Bit length mismatch. Original: {len(original_bits)}, Extracted: {len(extracted_bits)}")

    # Calculate bit errors
    bit_errors = np.sum(extracted_bits != original_bits)
    total_bits = len(original_bits)

    # Calculate BER
    ber = bit_errors / total_bits

    return ber, bit_errors, total_bits


def calculate_nc(original_bits, extracted_bits):
    original_bits = np.array(original_bits, dtype=np.float64)
    extracted_bits = np.array(extracted_bits, dtype=np.float64)

    if len(extracted_bits) != len(original_bits):
        min_len = min(len(extracted_bits), len(original_bits))
        extracted_bits = extracted_bits[:min_len]
        original_bits = original_bits[:min_len]

    numerator = np.sum(original_bits * extracted_bits)
    denominator = np.sqrt(np.sum(original_bits ** 2) * np.sum(extracted_bits ** 2))

    if denominator == 0:
        nc = 0.0
        print("Warning: Denominator is zero in NC calculation")
    else:
        nc = numerator / denominator

    return nc


def calculate_metrics(original_bits, extracted_bits, verbose=True):
    # Calculate BER
    ber, bit_errors, total_bits = calculate_ber(original_bits, extracted_bits)

    # Calculate NC
    nc = calculate_nc(original_bits, extracted_bits)

    # Additional metrics
    accuracy = 1 - ber
    correct_bits = total_bits - bit_errors

    metrics = {
        'ber': ber,
        'nc': nc,
        'accuracy': accuracy,
        'bit_errors': bit_errors,
        'correct_bits': correct_bits,
        'total_bits': total_bits
    }

    if verbose:
        print(f"\n{'='*60}")
        print(f"WATERMARK QUALITY METRICS (Pre-ECC)")
        print(f"{'='*60}")
        print(f"Bit Error Rate (BER)      : {ber:.3f} ({ber*100:.3f}%)")
        print(f"Normalized Correlation (NC): {nc:.3f}")
        print(f"Accuracy                  : {accuracy:.3f} ({accuracy*100:.3f}%)")
        print(f"Bit Errors                : {bit_errors}/{total_bits}")
        print(f"Correct Bits              : {correct_bits}/{total_bits}")
        print(f"{'='*60}\n")

        # Quality assessment
        if nc >= 0.95 and ber <= 0.05:
            quality = "Excellent"
        elif nc >= 0.85 and ber <= 0.15:
            quality = "Good"
        elif nc >= 0.70 and ber <= 0.30:
            quality = "Fair"
        else:
            quality = "Poor"

        print(f"Overall Quality: {quality}")

    return metrics

In [ ]:
def jpeg(image, quality=50):
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), quality]
    _, encoded_img = cv2.imencode('.jpg', image, encode_param)
    compressed_img = cv2.imdecode(encoded_img, cv2.IMREAD_COLOR)
    return compressed_img

def salt_pepper(image, salt_prob=0.01, pepper_prob=0.01):
    noisy = np.copy(image)
    total_pixels = image.size

    # Salt noise
    num_salt = np.ceil(salt_prob * total_pixels).astype(int)
    coords = [np.random.randint(0, i - 1, num_salt) for i in image.shape]
    noisy[tuple(coords)] = 255

    # Pepper noise
    num_pepper = np.ceil(pepper_prob * total_pixels).astype(int)
    coords = [np.random.randint(0, i - 1, num_pepper) for i in image.shape]
    noisy[tuple(coords)] = 0

    return noisy

def gaussian(image, mean=0, var=0.01):
    sigma = var ** 0.5
    gauss = np.random.normal(mean, sigma, image.shape).astype(np.float32)
    noisy = image.astype(np.float32) + gauss * 255  # Scale noise to pixel range
    noisy = np.clip(noisy, 0, 255).astype(np.uint8)
    return noisy

In [ ]:
attack1 = jpeg(watermarked_image, quality=90)
extract_jpeg1 = extract_pipeline_colored(attack1, embedding_info, levels=1)
if extract_jpeg1['success']:
  extracted_wm1 = extract_jpeg1['extracted_wm']
  cv2_imshow(extracted_wm1)

  # Calculate metrics1
  metrics1 = calculate_metrics(
    original_bits=embedding_info['original_data_bits'],
    extracted_bits=extract_jpeg1['extracted_bits']
  )
else:
  tampered_wm1 = extract_jpeg1['tampered_wm']
  cv2_imshow(tampered_wm1)
  metrics1 = calculate_metrics(
    original_bits=embedding_info['original_data_bits'],
    extracted_bits=extract_jpeg1['extracted_bits']
  )

try:
  psnr_val = calculate_psnr(host_img, attack1)
  print(f"PSNR: {psnr_val:.3f}")
  ssim_val, ssim_map = compute_ssim_safe(host_img, attack1)
  print(f"SSIM: {ssim_val:.3f}")
except Exception as e:
    print("SSIM compute error:", e)
cv2_imshow(attack1)

In [ ]:
attack2 = jpeg(watermarked_image, quality=80)
extract_jpeg2 = extract_pipeline_colored(attack2, embedding_info, levels=1)
if extract_jpeg2['success']:
  extracted_wm2 = extract_jpeg2['extracted_wm']
  cv2_imshow(extracted_wm2)

  metrics2 = calculate_metrics(
    original_bits=embedding_info['original_data_bits'],
    extracted_bits=extract_jpeg2['extracted_bits']
  )
else:
  tampered_wm2 = extract_jpeg2['tampered_wm']
  cv2_imshow(tampered_wm2)
  metrics2 = calculate_metrics(
    original_bits=embedding_info['original_data_bits'],
    extracted_bits=extract_jpeg2['extracted_bits']
  )

cv2_imshow(attack2)

In [ ]:
attack3 = jpeg(watermarked_image, quality=70)
extract_jpeg3 = extract_pipeline_colored(attack3, embedding_info, levels=1)

if extract_jpeg3['success']:
  extracted_wm3 = extract_jpeg3['extracted_wm']
  cv2_imshow(extracted_wm3)

  # Calculate metrics3
  metrics3 = calculate_metrics(
    original_bits=embedding_info['original_data_bits'],
    extracted_bits=extract_jpeg3['extracted_bits']
  )
else:
  tampered_wm3 = extract_jpeg3['tampered_wm']
  cv2_imshow(tampered_wm3)
  metrics3 = calculate_metrics(
    original_bits=embedding_info['original_data_bits'],
    extracted_bits=extract_jpeg3['extracted_bits']
  )

cv2_imshow(attack3)

In [ ]:
attack5 = jpeg(watermarked_image, quality=50)
extract_jpeg5 = extract_pipeline_colored(attack5, embedding_info, levels=1)

if extract_jpeg5['success']:
  extracted_wm5 = extract_jpeg5['extracted_wm']
  cv2_imshow(extracted_wm5)

  # Calculate metrics5
  metrics5 = calculate_metrics(
    original_bits=embedding_info['original_data_bits'],
    extracted_bits=extract_jpeg5['extracted_bits']
  )
else:
  tampered_wm5 = extract_jpeg5['tampered_wm']
  cv2_imshow(tampered_wm5)
  metrics5 = calculate_metrics(
    original_bits=embedding_info['original_data_bits'],
    extracted_bits=extract_jpeg5['extracted_bits']
  )

cv2_imshow(attack5)

In [ ]:
attack6 = jpeg(watermarked_image, quality=40)
extract_jpeg6 = extract_pipeline_colored(attack6, embedding_info, levels=1)

if extract_jpeg6['success']:
  extracted_wm6 = extract_jpeg6['extracted_wm']
  cv2_imshow(extracted_wm6)

  # Calculate metrics6
  metrics6 = calculate_metrics(
    original_bits=embedding_info['original_data_bits'],
    extracted_bits=extract_jpeg6['extracted_bits']
  )
else:
  tampered_wm6 = extract_jpeg6['tampered_wm']
  cv2_imshow(tampered_wm6)
  metrics6 = calculate_metrics(
    original_bits=embedding_info['original_data_bits'],
    extracted_bits=extract_jpeg6['extracted_bits']
  )

cv2_imshow(attack6)

In [ ]:
attack8 = gaussian(watermarked_image, 0, 0.0005)
extract_gaus1 = extract_pipeline_colored(attack8, embedding_info, levels=1)

if extract_gaus1['success']:
  extracted_wm8 = extract_gaus1['extracted_wm']
  cv2_imshow(extracted_wm8)

  # Calculate metrics8
  metrics8 = calculate_metrics(
    original_bits=embedding_info['original_data_bits'],
    extracted_bits=extract_gaus1['extracted_bits']
  )
else:
  tampered_wm8 = extract_gaus1['tampered_wm']
  cv2_imshow(tampered_wm8)
  metrics8 = calculate_metrics(
    original_bits=embedding_info['original_data_bits'],
    extracted_bits=extract_gaus1['extracted_bits']
  )

cv2_imshow(attack8)

In [ ]:
attack9 = gaussian(watermarked_image, 0, 0.001)
extract_gaus2 = extract_pipeline_colored(attack9, embedding_info, levels=1)

if extract_gaus2['success']:
  extracted_wm9 = extract_gaus2['extracted_wm']
  cv2_imshow(extracted_wm9)

  # Calculate metrics9
  metrics9 = calculate_metrics(
    original_bits=embedding_info['original_data_bits'],
    extracted_bits=extract_gaus2['extracted_bits']
  )
else:
  tampered_wm9 = extract_gaus2['tampered_wm']
  cv2_imshow(tampered_wm9)
  metrics9 = calculate_metrics(
    original_bits=embedding_info['original_data_bits'],
    extracted_bits=extract_gaus2['extracted_bits']
  )

cv2_imshow(attack9)

In [ ]:
attack10 = gaussian(watermarked_image, 0, 0.005)
extract_gaus3 = extract_pipeline_colored(attack10, embedding_info, levels=1)

if extract_gaus3['success']:
  extracted_wm10 = extract_gaus3['extracted_wm']
  cv2_imshow(extracted_wm10)

  # Calculate metrics10
  metrics10 = calculate_metrics(
    original_bits=embedding_info['original_data_bits'],
    extracted_bits=extract_gaus3['extracted_bits']
  )
else:
  tampered_wm10 = extract_gaus3['tampered_wm']
  cv2_imshow(tampered_wm10)
  metrics10 = calculate_metrics(
    original_bits=embedding_info['original_data_bits'],
    extracted_bits=extract_gaus3['extracted_bits']
  )

cv2_imshow(attack10)

In [ ]:
attack11 = salt_pepper(watermarked_image, 0.001, 0.001)
extract_sp1 = extract_pipeline_colored(attack11, embedding_info, levels=1)

if extract_sp1['success']:
  extracted_wm11 = extract_sp1['extracted_wm']
  cv2_imshow(extracted_wm11)

  # Calculate metrics11
  metrics11 = calculate_metrics(
    original_bits=embedding_info['original_data_bits'],
    extracted_bits=extract_sp1['extracted_bits']
  )
else:
  tampered_wm11 = extract_sp1['tampered_wm']
  cv2_imshow(tampered_wm11)
  metrics11 = calculate_metrics(
    original_bits=embedding_info['original_data_bits'],
    extracted_bits=extract_sp1['extracted_bits']
  )

cv2_imshow(attack11)

In [ ]:
attack12 = salt_pepper(watermarked_image, 0.005, 0.005)
extract_sp2 = extract_pipeline_colored(attack12, embedding_info, levels=1)

if extract_sp2['success']:
  extracted_wm12 = extract_sp2['extracted_wm']
  cv2_imshow(extracted_wm12)

  # Calculate metrics12
  metrics12 = calculate_metrics(
    original_bits=embedding_info['original_data_bits'],
    extracted_bits=extract_sp2['extracted_bits']
  )
else:
  tampered_wm12 = extract_sp2['tampered_wm']
  cv2_imshow(tampered_wm12)
  metrics12 = calculate_metrics(
    original_bits=embedding_info['original_data_bits'],
    extracted_bits=extract_sp2['extracted_bits']
  )

cv2_imshow(attack12)

In [ ]:
attack13 = salt_pepper(watermarked_image, 0.01, 0.01)
extract_sp3 = extract_pipeline_colored(attack13, embedding_info, levels=1)

if extract_sp3['success']:
  extracted_wm13 = extract_sp3['extracted_wm']
  cv2_imshow(extracted_wm13)

  # Calculate metrics13
  metrics13 = calculate_metrics(
    original_bits=embedding_info['original_data_bits'],
    extracted_bits=extract_sp3['extracted_bits']
  )
else:
  tampered_wm13 = extract_sp3['tampered_wm']
  cv2_imshow(tampered_wm13)
  metrics13 = calculate_metrics(
    original_bits=embedding_info['original_data_bits'],
    extracted_bits=extract_sp3['extracted_bits']
  )

cv2_imshow(attack13)